In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt

    
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                          save_calibration_data, get_footprints_from_suite2p)


#from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
#                         correct_drift_single_frame, template_generation, 
 #                        plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0

from utils.calcium import calcium


Operating system:  Linux


Autosaving every 180 seconds


2023-01-20 12:45:51.779089: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-01-20 12:45:51.844009: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-01-20 12:45:51.845978: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/cat/.conda/envs/bmi/lib/python3.8/site-packages/cv2/../../lib64:
2023-01-20 12:45:51.845

In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################

#
#fname = r'F:\bmi\andres\20230115\M1\calibration\Image_001_001.raw'
fname = '/media/cat/4TB/donato/andres/DON011733/20230118/calibration/Image_001_001.raw'

# 
bmi_c = CalibrationTools(fname)
bmi_c.calcium = calcium
#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is fast in other implemenations

#
bmi_c.std_map = bmi_c.data[:1000].mean(0)

# load suite2p footprints
bmi_c = get_footprints_from_suite2p(bmi_c)


print ("DONE...")

memmap :  (20000, 512, 512)
  Fluorescence data loading information
         sample rate:  30 hz
         self.F (fluorescence):  (93, 20000)
         self.Fneu (neuropile):  (93, 20000)
         self.iscell (Suite2p cell classifier output):  (306, 2)
              of which number of good cells:  (93,)
         self.spks (deconnvoved spikes):  (93, 20000)
         self.stat (footprints structure):  (93,)
         mean std over all cells :  57.0
# of footprints;  93
DONE...


In [10]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
order_type = 'snr'  # 'snr' or 'f0'
bmi_c.compute_roi_traces_f0_and_reorder_cells(order_type)  # this function also coputes the snr / f0s of the cells

#
bmi_c.scale=3  
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 10       # Subsample the time series to go faster;

# visualize traces
bmi_c.vmin=0
bmi_c.vmax=5000
bmi_c.visualize_traces_snr_order(bmi_c.std_map)

# 


computing roi traces for SNR indexing: 100%|█████████████████| 2000/2000 [00:01<00:00, 1561.38it/s]


memmap :  (20000, 512, 512)


In [64]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [0,1]
bmi_c.ensemble2 = [7,13]
bmi_c.both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
print ("all cells:", bmi_c.both)

#
bmi_c.show_traces_ids(bmi_c.both)


all cells: [ 0  1  7 13]


In [65]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles(bmi_c.std_map)

print ("DONE...")

100%|█████████████████████████████████████████████████████| 20000/20000 [00:00<00:00, 26820.12it/s]


DONE...


In [66]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 10   # reward lockout in seconds
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

#
# find 30% reward threshold
bmi_c.reward_rate = 0.25#*0.85

#gr.find_reward_thresholds_low_and_high()
bmi_c.find_reward_thresholds_high()  # this only rewards when sound passes specific level

#
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()


#
bmi_c.reward_rate_scaling_factor = 1.0

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


COMPUTED # of roi traces:  93


100%|███████████████████████████████████████████████████| 19995/19995 [00:00<00:00, 1784629.81it/s]

low, high:  -1.8821478090623893 2.506148934096097
nsec recording:  666 max # of random rewards (i.e. every 30sec)  22
 @30% reward:  5
updated rewards #:  5 -1.0705658474726232 1.4254977449724109
thresholds:  1.4254977449724109
 PROCESSING...
bmi_c.high: scaled by:  1.0 , final value:   1.4254977449724109


In [251]:
#############################################
########### SAVE DATA #######################
#############################################

#    
save_calibration_data(bmi_c)  

AttributeError: 'CalibrationTools' object has no attribute 'std_map'

In [49]:
data = np.load('/media/cat/4TBSSD/donato/DON8498/22-06-08/rois_pixels_and_thresholds.npz',
               allow_pickle=True)

ensemble1_footprints = data['ensemble1_footprints']
contours_local = data['ensemble1_contours']

FileNotFoundError: [Errno 2] No such file or directory: '/media/cat/4TBSSD/donato/DON8498/22-06-08/rois_pixels_and_thresholds.npz'

In [4]:
cell_id =1
plt.figure()
temp = bmi_c.roi_traces[cell_id]
plt.plot((temp - np.median(temp))/np.median(temp))
plt.plot((temp - bmi_c.roi_f0s[cell_id])/bmi_c.roi_f0s[cell_id])
plt.show()

NameError: name 'bmi_c' is not defined

In [6]:
plt.figure()
for c in range(len(contours_local)):
    for k in range(len(contours_local[c])-1):

        plt.plot([contours_local[c][k][0],
                            contours_local[c][k+1][0]],
                           [contours_local[c][k][1],
                            contours_local[c][k+1][1]],
                            c='blue',
                            linewidth=5)
plt.show()


In [16]:
data1 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_regular_filter.npz',
                allow_pickle=True)
e1 = data1['ensemble_activity']
print (e1.shape)


data2 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_5timesteps.npz',
                allow_pickle=True)
e2 = data2['ensemble_activity']
print (e2.shape)

data3 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_no_filter.npz',
                allow_pickle=True)
e3 = data3['ensemble_activity']
print (e3.shape)

data4 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results.npz',
                allow_pickle=True)
e4 = data4['ensemble_activity']
print (e4.shape)


#############################################################
plt.figure()

t = np.arange(e1.shape[1])/30
plt.plot(t,
         e1[0]-e1[1],
         c='blue', 
         linewidth = 4,
         label='current filter')
plt.plot(t,
         e2[0]-e2[1],
         linewidth = 4,
         c='red', label = 'mean last 5 time steps')
plt.plot(t,
         e3[0]-e3[1],
         linewidth = 4,
         c='green', label = 'no filter')
plt.plot(t,
         e4[0]-e4[1],
         linewidth = 4,
         c='black', label = 'graded filter last 5 time steps')


plt.xlabel("time (sec)", fontsize=16)
plt.ylabel("Ensemble 1 - Ensemble 2", fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.plot([t[0],t[-1]],
         [0,0],
         '--',
        c='black')
plt.legend()
plt.show()

(2, 3000)
(2, 3000)
(2, 3000)
(2, 3000)


In [3]:
def get_octave_frequencies(low_freq,
						   high_freq,
						   octave_size=0.25):
	#
	octaves = []

	#
	octaves.append(low_freq)
	temp = low_freq
	while True:
		temp = temp * (1 + octave_size)
		if temp > high_freq:
			break
		octaves.append(temp)
	"""
	low_freq = 1000
	high_freq = 16000
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
    
	octaves = 2**(octave_size*x)
    
	#
	return np.array(1000*octaves)
	"""
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
	octaves = 2**(octave_size*x)
	return np.array(1000*octaves)

	#

In [4]:
low_freq = 2000
high_freq = 16000

octaves = get_octave_frequencies(low_freq,high_freq,octave_size=0.25)

NameError: name 'x' is not defined